# 08 — Fine-tuning eficiente con LoRA / PEFT (opcional)

> **Objetivo.** Escalar a `t5-base` (220M params) en hardware modesto entrenando solo ~0.5% de los parámetros vía **LoRA** (Low-Rank Adaptation).

## Por qué LoRA

Entrenar todos los pesos de `t5-base` en MPS:
- Memoria: ~3-4 GB solo para los pesos en bf16 + ~6-8 GB para optimizer states (AdamW guarda 2 momentos por parámetro).
- Tiempo: ~2-3× lento vs `t5-small`.

**LoRA** (Hu et al., 2021) congela el modelo y entrena **adaptadores de bajo rango** (matrices `B·A` con `rank << dim`) que se inyectan en `q_proj`/`v_proj`. Resultado:

- ~0.5% de los parámetros entrenables.
- Optimizer states proporcionalmente pequeños.
- Adaptador serializable (~5 MB) — puedes tener muchos adaptadores por modelo base.
- Calidad ~95% del full fine-tune.

```
         W (frozen, 768x768)
x ──────►[····················]──────► y
          ▲                  ▲
          │  +  B·A (rank=8) │
          │   B (768x8)      │
          │   A (8x768)      │
          ▲ trainable ───────┘
```

## Trade-offs

| Aspecto | Full FT | LoRA |
|---|---|---|
| Params entrenables | 100% | ~0.5% |
| RAM optim states | Alta | Baja |
| Calidad | Mejor | ~5% peor |
| Inferencia | Igual | Igual (merge opcional) |
| Múltiples tareas | 1 modelo cada una | 1 base + N adaptadores |


In [ ]:
# === CONFIG ===
BASE_MODEL    = "google-t5/t5-base"   # sube a t5-large si tienes 32+ GB
LORA_R        = 8
LORA_ALPHA    = 16
LORA_DROPOUT  = 0.05
TARGET_MODULES = ["q", "v"]            # T5 attention proj names

BATCH_SIZE    = 4
GRAD_ACCUM    = 4
LR            = 1e-3                   # LoRA tolera LR mayor
NUM_EPOCHS    = 3
SEED          = 42
# ==============

import os, torch
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

from pathlib import Path
from datasets import load_from_disk
from transformers import (
    AutoTokenizer, T5ForConditionalGeneration,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq, set_seed,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

set_seed(SEED)

ROOT       = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
TOKENIZED  = ROOT / "artifacts" / "tokenized_dataset"
OUTPUT_DIR = ROOT / "artifacts" / "t5-base-lora"

device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "| Base:", BASE_MODEL)

## 1. Cargar tokenizer + modelo base + dataset

Nota: el dataset tokenizado del notebook 04 fue construido con el tokenizer de `t5-small`. Los tokenizers de t5-small/base/large son **compatibles** (mismo SentencePiece y vocab). Si por algún motivo cambias de familia (a `flan-t5`, p.ej.) debes re-tokenizar.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
base_model = T5ForConditionalGeneration.from_pretrained(BASE_MODEL)

total = sum(p.numel() for p in base_model.parameters())
print(f"Modelo base: {BASE_MODEL}  → {total/1e6:.1f}M params")

## 2. Configurar LoRA

Parámetros clave:
- **`r`** — rango de los adaptadores. 4-16 funciona bien para T5.
- **`lora_alpha`** — escalado (intuitivamente: tasa de aprendizaje efectiva del adaptador). Típicamente `2*r`.
- **`target_modules`** — qué proyecciones reemplazar. En T5 son `q`, `k`, `v`, `o` dentro de cada bloque attention. Se suele afectar `q` y `v` (paper LoRA original).
- **`task_type=SEQ_2_SEQ_LM`** — necesario para que PEFT exponga métodos como `generate`.

In [ ]:
lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
)

model = get_peft_model(base_model, lora_cfg)
model.print_trainable_parameters()

Esa salida muestra algo como `trainable params: 884,736 || all params: 222,887,424 || trainable%: 0.397`. Solo entrenamos ~0.4% del modelo.

## 3. Trainer (mismo patrón que notebook 05)

In [ ]:
import numpy as np
import evaluate
bleu = evaluate.load("sacrebleu")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple): preds = preds[0]
    preds  = np.where(preds  != -100, preds,  tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds  = [p.strip() for p in tokenizer.batch_decode(preds, skip_special_tokens=True)]
    decoded_labels = [[l.strip()] for l in tokenizer.batch_decode(labels, skip_special_tokens=True)]
    return {"sacrebleu": round(bleu.compute(predictions=decoded_preds, references=decoded_labels)["score"], 4)}

In [ ]:
tokenized = load_from_disk(str(TOKENIZED))
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True, label_pad_token_id=-100, pad_to_multiple_of=8)

args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),
    overwrite_output_dir=True,

    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,

    learning_rate=LR,
    weight_decay=0.0,
    warmup_ratio=0.05,

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="sacrebleu",
    greater_is_better=True,

    predict_with_generate=True,
    generation_max_length=128,
    generation_num_beams=4,

    bf16=True,
    fp16=False,
    logging_steps=50,
    report_to="none",
    seed=SEED,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

## 4. Guardar el adaptador (no el modelo entero)

LoRA solo persiste los pesos del adaptador (~5-30 MB). Para servir, cargas el base model + el adaptador encima.

In [ ]:
ADAPTER = OUTPUT_DIR / "final-adapter"
trainer.model.save_pretrained(str(ADAPTER))
tokenizer.save_pretrained(str(ADAPTER))
print("Adaptador guardado en:", ADAPTER)

# Tamaño en disco
import subprocess
result = subprocess.run(["du", "-sh", str(ADAPTER)], capture_output=True, text=True)
print(result.stdout.strip())

## 5. Cargar el adaptador para inferencia

Dos formas:

### Opción A — modelo + adaptador separados (default)
```python
base   = T5ForConditionalGeneration.from_pretrained("google-t5/t5-base")
model  = PeftModel.from_pretrained(base, str(ADAPTER))
```

### Opción B — merge adapter into base (inferencia más rápida, no se puede combinar otros adaptadores después)
```python
merged = model.merge_and_unload()
merged.save_pretrained("./t5-base-merged")
```

In [ ]:
# Demostración: cargar y probar
del model, trainer
torch.mps.empty_cache() if device == "mps" else None

base = T5ForConditionalGeneration.from_pretrained(BASE_MODEL).to(device)
infer_model = PeftModel.from_pretrained(base, str(ADAPTER)).to(device)
infer_model.eval()

tests = [
    "He go to school every day.",
    "I has a apple.",
    "This is a correct sentence.",
]
for t in tests:
    enc = tokenizer("grammar: " + t, return_tensors="pt").to(device)
    with torch.no_grad():
        out = infer_model.generate(**enc, max_new_tokens=64, num_beams=4)
    print(f"IN : {t}")
    print(f"OUT: {tokenizer.decode(out[0], skip_special_tokens=True)}\n")

## 6. Comparativa final

| | t5-small full FT (notebook 05) | t5-base + LoRA (este notebook) |
|---|---|---|
| Params totales | 60M | 220M |
| Params entrenados | 60M (100%) | ~880K (0.4%) |
| RAM pico (MPS, batch=8) | ~3 GB | ~5 GB |
| Tamaño checkpoint | ~240 MB | ~5 MB (+ base reusable) |
| Tiempo / epoch (M1 Pro 16GB) | ~7 min | ~12 min |
| SacreBLEU esperado | base | +2-4 puntos |

## Cuándo usar cada uno

- **Full fine-tuning small** — PoC rápida, hardware limitado, tarea única.
- **LoRA sobre base/large** — más calidad, múltiples tareas, infraestructura ML madura.
- **LoRA sobre `flan-t5-large`** — el sweet spot moderno para muchos usos.

---

Fin de la PoC. Para profundizar: ver `docs/decisiones_tecnicas.md` y `docs/arquitectura.md`.
